## 0 - Imports


In [1]:
# MUST RUN THIS FIRST - Disable torch.compile BEFORE importing moshi
import torch
import torch._dynamo

# Completely disable torch compilation
torch._dynamo.config.suppress_errors = True
torch.set_float32_matmul_precision('high')

original_compile = torch.compile
def no_compile(model, *args, **kwargs):
    print("torch.compile disabled - using eager mode")
    return model

torch.compile = no_compile

print("Torch compilation disabled")

Torch compilation disabled


In [2]:
import torch
# import torchaudio
import numpy as np
import soundfile as sf
# import librosa
import sounddevice as sd
import pandas as pd
import sys
# import time
import os
from pathlib import Path
import IPython.display as ipd

#fix filepathing
# sys.path.append(r"C:\Users\jking36\Documents\Master\Capstone\src\moshi\moshi")
#moshi imports
from moshi.models.loaders import CheckpointInfo
from moshi.models.tts import TTSModel, DEFAULT_DSM_TTS_REPO, DEFAULT_DSM_TTS_VOICE_REPO


print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.9.1+cu128
CUDA available: True


/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 - Creation of TTS Logic

In [6]:
print(sys.path)

print(os.getcwd())

print(sys.executable, '\n---> path for env')

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages', '/tmp/tmpa8peezk5']
/home/exouser/Language-Project/notebooks
/home/exouser/Language-Project-1/Moshi2/bin/python 
---> path for env


In [3]:
# Set device (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the TTS model
print("Loading model...")
checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)
tts_model = TTSModel.from_checkpoint_info(
    checkpoint_info,
    n_q=32,  # Number of codebook quantizers
    temp=0.6,  # Temperature for generation
    device=device,
    dtype=torch.float16 if device.type == 'cuda' else torch.float32
)

print("Model loaded successfully!")
print(f"See https://huggingface.co/{DEFAULT_DSM_TTS_VOICE_REPO} for available voices.")

Using device: cuda
Loading model...
Model loaded successfully!
See https://huggingface.co/kyutai/tts-voices for available voices.


In [4]:
# Text To Speech Demo - 45-70sec cpu ----> GPU ~6sec
text = "University of Arizona is pretty cool! Testing other languages in romanized characters, arigato gozaimasu. Ego deki masuka"
voice = "vctk/p228_023.wav"  # specify voice from huggingface

voice_path = tts_model.get_voice_path(voice)

print("Generating audio...")
audio_list = tts_model.simple_generate(
    text=text,
    voice=voice,
    cfg_coef=2.0
)

# It returns a list - get the first element
audio_tensor = audio_list[0]

# Convert to numpy
audio = audio_tensor.cpu().numpy()
if audio.ndim > 1:
    audio = audio[0]  # Get first channel if needed

audio = np.clip(audio, -1, 1)

print(f"Generated {len(audio)/24000:.2f} seconds of audio")

# Play it
ipd.Audio(audio, rate=24000)

Generating audio...


Generating: 0it [00:00, ?it/s]

torch.compile disabled - using eager mode
torch.compile disabled - using eager mode
torch.compile disabled - using eager mode


Generating: 119it [00:04, 26.78it/s]


Generated 9.36 seconds of audio


## 2 - TTS Function

In [14]:
def MoshiTTS(text):
    # Set device (use GPU if available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load the TTS model
    print("Loading model...")
    checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)
    tts_model = TTSModel.from_checkpoint_info(
        checkpoint_info,
        n_q=32,  # Number of codebook quantizers
        temp=0.6,  # Temperature for generation
        device=device,
        dtype=torch.float16 if device.type == 'cuda' else torch.float32
    )

    print("Model loaded successfully!")
    print(f"See https://huggingface.co/{DEFAULT_DSM_TTS_VOICE_REPO} for available voices.")

    ####### Audio Creation ######

    # Text To Speech Demo - 45-70sec
    # text = "University of Arizona is pretty cool! Testing other languages in romanized characters, arigato gozaimasu. Ego deki masuka"
    voice = "vctk/p228_023.wav"  # specify voice from huggingface

    voice_path = tts_model.get_voice_path(voice)

    print("Generating audio...")
    audio_list = tts_model.simple_generate(
        text=text,
        voice=voice,
        cfg_coef=2.0
    )

    # It returns a list - get the first element
    audio_tensor = audio_list[0]

    # Convert to numpy
    audio = audio_tensor.cpu().numpy()
    if audio.ndim > 1:
        audio = audio[0]  # Get first channel if needed

    audio = np.clip(audio, -1, 1)

    print(f"Generated {len(audio)/24000:.2f} seconds of audio")

    # Play it
    return ipd.Audio(audio, rate=24000)

In [15]:
MoshiTTS('Testing the Moshi Text to speech function with input text, next up streaming transcription and translation') #GPU~5sec

Using device: cuda
Loading model...
Model loaded successfully!
See https://huggingface.co/kyutai/tts-voices for available voices.
Generating audio...


Generating: 0it [00:00, ?it/s]

torch.compile disabled - using eager mode
torch.compile disabled - using eager mode
torch.compile disabled - using eager mode


Generating: 87it [00:03, 22.94it/s]


Generated 6.80 seconds of audio


In [22]:
# Let's see what the TTSModel actually has
print("TTSModel methods:")
print([m for m in dir(tts_model) if not m.startswith('_') and callable(getattr(tts_model, m))])

# Also check if there's an LM model inside
print("\nTTSModel attributes:")
print([a for a in dir(tts_model) if not a.startswith('_')])

# Check if there's something like lm_gen or audio_lm
if hasattr(tts_model, 'lm_gen'):
    print(f"\nlm_gen type: {type(tts_model.lm_gen)}")
    print(f"lm_gen methods: {[m for m in dir(tts_model.lm_gen) if 'audio' in m.lower() or 'decode' in m.lower()]}")

TTSModel methods:
['from_checkpoint_info', 'generate', 'get_prefix', 'get_voice_path', 'lm', 'make_condition_attributes', 'mimi', 'prepare_script', 'simple_generate', 'warmup']

TTSModel attributes:
['cfg_coef', 'delay_steps', 'final_padding', 'from_checkpoint_info', 'generate', 'get_prefix', 'get_voice_path', 'lm', 'machine', 'make_condition_attributes', 'max_gen_length', 'max_speakers', 'mimi', 'multi_speaker', 'multistream', 'n_q', 'padding_bonus', 'prepare_script', 'simple_generate', 'temp', 'tokenizer', 'valid_cfg_conditionings', 'voice_repo', 'voice_suffix', 'warmup']


In [ ]:
# Save to file
import soundfile as sf

output_path = "../audio/output_tts.wav"
sf.write(output_path, audio, 24000)
print(f"Audio saved to {output_path}")

## 3 - Streaming STT Logic
- Attempt Low Latency 2-5sec delay
- NEED GPU - uses 100% CPU and does not stream

In [18]:
from moshi.models import LMGen
# load 1b parameter kyutai model
checkpoint_info = CheckpointInfo.from_hf_repo('kyutai/stt-1b-en_fr')
device = 'cuda' # or 'cuda' with GPU
mimi = checkpoint_info.get_mimi(device=device)
moshi = checkpoint_info.get_moshi(device = device)
print('Model Loaded!')

#tokenizer
text_tokenizer = checkpoint_info.get_text_tokenizer()

#LMGen for streaming generation
lm_gen = LMGen(moshi,temp = 0.8, temp_text = 0.8)

# Audio Set Up
SAMPLE_RATE = 24000 # model needs 24kHz
FRAME_SIZE = int(SAMPLE_RATE/12.5) #mimi processes at 12.5Hz

#Max Recording time
MAX_TIME = 120 #sec

# Buffer to accumulate audio
audio_buffer = []

# Start streaming context
streaming_context = lm_gen.streaming(1).__enter__()

def audio_callback(indata, frames, time, status):
    if status:
        # print(f'Status: {status}')
        print('')
    
    try:
        # DEBUG: Print shapes
        # print(f"DEBUG: indata.shape = {indata.shape}")
        
        # Add audio to buffer (extract mono channel)
        mono_audio = indata[:, 0].copy()
        # print(f"DEBUG: mono_audio.shape = {mono_audio.shape}")
        
        audio_buffer.append(mono_audio)
        
        # Process when we have enough audio for one frame
        total_samples = sum(len(chunk) for chunk in audio_buffer)
        # print(f"DEBUG: total_samples in buffer = {total_samples}")
        
        if total_samples >= FRAME_SIZE:
            # Concatenate buffered audio
            audio_chunk = np.concatenate(audio_buffer)
            # print(f"DEBUG: audio_chunk.shape after concat = {audio_chunk.shape}")
            # print(f"DEBUG: audio_chunk is 1D? {audio_chunk.ndim == 1}")
            
            # Take exactly FRAME_SIZE samples
            audio_frame = audio_chunk[:FRAME_SIZE]  # THIS IS THE CORRECT LINE
            # print(f"DEBUG: audio_frame.shape = {audio_frame.shape}")
            
            # Keep leftover samples
            leftover = audio_chunk[FRAME_SIZE:]
            audio_buffer.clear()
            if len(leftover) > 0:
                audio_buffer.append(leftover)
            
            # Convert to tensor [1, 1, T]
            audio_tensor = torch.from_numpy(audio_frame).float()
            audio_tensor = audio_tensor.unsqueeze(0).unsqueeze(0)
            # print(f"DEBUG: audio_tensor.shape = {audio_tensor.shape}")
            
            #Enconde with Mimi
            with torch.no_grad():
                codes = mimi.encode(audio_tensor)
                # Generate with LMGen
                tokens_out = lm_gen.step(codes)
                
                if tokens_out is not None:
                    # Extract text tokens (first stream)
                    text_tokens = tokens_out[0, 0, :]
                    
                    # Decode to text
                    text = text_tokenizer.decode(text_tokens.tolist())
                    if text.strip():
                        print(f'Transcription: {text}', flush=True)


    except Exception as e:
        print(f'Error: {e}')
        import traceback
        traceback.print_exc()

print(f'Recording at {SAMPLE_RATE}Hz...')
print(f'Maximum Recording Time --> {MAX_TIME} sec')
print('Start Speaking! Press enter to stop')

try:
    with sd.InputStream(
        callback=audio_callback,
        samplerate=SAMPLE_RATE, 
        channels=1,
        dtype='float32'
    ):
        input() #Press enter to stop

except Exception as e:
    print(f'\nException: {e}')
finally:
    streaming_context.__exit__(None,None,None)
    print('\nRecording session ended')
    




Model Loaded!
Recording at 24000Hz...
Maximum Recording Time --> 120 sec
Start Speaking! Press enter to stop
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)


Traceback (most recent call last):
  File "/tmp/ipykernel_131636/89418264.py", line 70, in audio_callback
    codes = mimi.encode(audio_tensor)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 386, in encode
    emb = self._encode_to_unquantized_latent(x)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 359, in _encode_to_unquantized_latent
    emb = self.encoder(x)
          ^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    else:
          
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    or _global_forward_hooks or _global_forward_pre_hooks):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-

Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)


Traceback (most recent call last):
  File "/tmp/ipykernel_131636/89418264.py", line 70, in audio_callback
    codes = mimi.encode(audio_tensor)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 386, in encode
    emb = self._encode_to_unquantized_latent(x)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 359, in _encode_to_unquantized_latent
    emb = self.encoder(x)
          ^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    else:
          
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    or _global_forward_hooks or _global_forward_pre_hooks):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-

Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)


...

Traceback (most recent call last):
  File "/tmp/ipykernel_131636/89418264.py", line 70, in audio_callback
    codes = mimi.encode(audio_tensor)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 386, in encode
    emb = self._encode_to_unquantized_latent(x)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 359, in _encode_to_unquantized_latent
    emb = self.encoder(x)
          ^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    else:
          
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    or _global_forward_hooks or _global_forward_pre_hooks):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-

Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)


Traceback (most recent call last):
  File "/tmp/ipykernel_131636/89418264.py", line 70, in audio_callback
    codes = mimi.encode(audio_tensor)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 386, in encode
    emb = self._encode_to_unquantized_latent(x)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 359, in _encode_to_unquantized_latent
    emb = self.encoder(x)
          ^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    else:
          
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    or _global_forward_hooks or _global_forward_pre_hooks):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-

Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)
Error: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)

Recording session ended


Traceback (most recent call last):
  File "/tmp/ipykernel_131636/89418264.py", line 70, in audio_callback
    codes = mimi.encode(audio_tensor)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 386, in encode
    emb = self._encode_to_unquantized_latent(x)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/src/moshi/moshi/moshi/models/compression.py", line 359, in _encode_to_unquantized_latent
    emb = self.encoder(x)
          ^^^^^^^^^^^^^^^
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    else:
          
  File "/home/exouser/Language-Project-1/Moshi2/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    or _global_forward_hooks or _global_forward_pre_hooks):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exouser/Language-

In [ ]:
import numpy as np
import torch
from fastapi import FastAPI, WebSocket
from fastapi.staticfiles import StaticFiles

app = FastAPI()

# Example model load
# model = load_model().to("cuda")

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    print("Browser connected")

    while True:
        audio_bytes = await websocket.receive_bytes()
        audio = np.frombuffer(audio_bytes, dtype=np.float32)

        audio_tensor = torch.tensor(audio).to("cuda")

        # prediction = model(audio_tensor)
        prediction = "dummy transcription"

        await websocket.send_text(prediction)

app.mount("/", StaticFiles(directory="static", html=True), name="static")

In [7]:
import numpy as np
import torch
import sounddevice as sd
from moshi.models import LMGen
from moshi.models.loaders import CheckpointInfo

checkpoint_info = CheckpointInfo.from_hf_repo('kyutai/stt-1b-en_fr')
device = 'cuda'
mimi = checkpoint_info.get_mimi(device=device)
moshi = checkpoint_info.get_moshi(device=device)
print('Model Loaded!')

text_tokenizer = checkpoint_info.get_text_tokenizer()
lm_gen = LMGen(moshi, temp=0.8, temp_text=0.8)

SAMPLE_RATE = 24000
FRAME_SIZE = int(SAMPLE_RATE / 12.5)
MAX_TIME = 120

audio_buffer = []

streaming_context = lm_gen.streaming(1).__enter__()

def audio_callback(indata, frames, time, status):
    try:
        mono_audio = indata[:, 0].copy()
        audio_buffer.append(mono_audio)

        total_samples = sum(len(chunk) for chunk in audio_buffer)

        if total_samples >= FRAME_SIZE:
            audio_chunk = np.concatenate(audio_buffer)
            audio_frame = audio_chunk[:FRAME_SIZE]
            leftover = audio_chunk[FRAME_SIZE:]
            audio_buffer.clear()
            if len(leftover) > 0:
                audio_buffer.append(leftover)

            audio_tensor = torch.from_numpy(audio_frame).float()
            audio_tensor = audio_tensor.unsqueeze(0).unsqueeze(0).to(device)  # CHANGE 1: .to(device)

            with torch.no_grad():
                codes = mimi.encode(audio_tensor)
                tokens_out = lm_gen.step(codes)

            if tokens_out is not None:
                for token_id in tokens_out[0, 0, :].tolist():  # CHANGE 2: decode token by token
                    if token_id not in (0, 3):
                        text = text_tokenizer.decode([token_id])
                        if text:
                            print(text, end='', flush=True)

    except Exception as e:
        print(f'Error: {e}')
        import traceback
        traceback.print_exc()

print(f'Recording at {SAMPLE_RATE}Hz...')
print('Start Speaking! Press the Jupyter stop button (■) to stop.')

try:
    with sd.InputStream(
        callback=audio_callback,
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype='float32',
        blocksize=FRAME_SIZE  # CHANGE 3: fixed blocksize so buffer logic is simpler
    ):
        import time
        for _ in range(int(MAX_TIME * 10)):
            time.sleep(0.1)  # CHANGE 4: loop instead of input() so Jupyter doesn't freeze
except KeyboardInterrupt:
    pass
finally:
    streaming_context.__exit__(None, None, None)
    print('\nRecording session ended')

Model Loaded!
Recording at 24000Hz...
Start Speaking! Press the Jupyter stop button (■) to stop.

Recording session ended


In [8]:
import numpy as np
import torch
from fastapi import FastAPI, WebSocket
from moshi.models import LMGen
from moshi.models.loaders import CheckpointInfo
import uvicorn
import threading

SAMPLE_RATE = 24000
FRAME_SIZE  = int(SAMPLE_RATE / 12.5)
DEVICE      = 'cuda'

print('Loading model...')
checkpoint_info = CheckpointInfo.from_hf_repo('kyutai/stt-1b-en_fr')
mimi           = checkpoint_info.get_mimi(device=DEVICE)
moshi_model    = checkpoint_info.get_moshi(device=DEVICE)
text_tokenizer = checkpoint_info.get_text_tokenizer()
print('Model loaded!')

Loading model...
Model loaded!


In [ ]:
app = FastAPI()

@app.websocket('/transcribe')
async def transcribe(websocket: WebSocket):
    await websocket.accept()
    print('Client connected.')
    lm_gen = LMGen(moshi_model, temp=0.8, temp_text=0.8)

    dummy = torch.zeros(1, 1, FRAME_SIZE).to(DEVICE)
    with lm_gen.streaming(1):
        with torch.no_grad():
            lm_gen.step(mimi.encode(dummy))

    with lm_gen.streaming(1):
        try:
            while True:
                raw    = await websocket.receive_bytes()
                frame  = np.frombuffer(raw, dtype=np.float32)
                if len(frame) != FRAME_SIZE:
                    continue
                tensor = torch.from_numpy(frame).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    tokens_out = lm_gen.step(mimi.encode(tensor))
                if tokens_out is not None:
                    for tid in tokens_out[0, 0, :].tolist():
                        if tid not in (0, 3):
                            text = text_tokenizer.decode([tid])
                            if text:
                                await websocket.send_text(text)
        except Exception as e:
            print(f'Client disconnected: {e}')

# Run FastAPI in background thread so notebook doesn't freeze
thread = threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True)
thread.start()
print('Server running on port 8000.')

# Show mic UI directly in the notebook output
from IPython.display import display, HTML
JETSTREAM_IP = 'localhost'  # <-- INPUT FROM JETSTREAM2 -pub

display(HTML(f'''
<button onclick="startTranscription()" style="padding:10px 20px;background:#2ecc71;color:white;border:none;border-radius:6px;font-size:1em;cursor:pointer">
    Start Transcription
</button>
<button onclick="stopTranscription()" style="padding:10px 20px;background:#e74c3c;color:white;border:none;border-radius:6px;font-size:1em;cursor:pointer">
    Stop
</button>
<p id="status" style="color:#888">Not connected.</p>
<div id="output" style="border:1px solid #ccc;padding:20px;min-height:150px;border-radius:8px;font-size:1.1em;line-height:1.6;margin-top:10px"></div>

<script>
const SAMPLE_RATE = 24000;
const FRAME_SIZE  = Math.floor(SAMPLE_RATE / 12.5);
const WS_URL      = 'ws://{JETSTREAM_IP}:8000/transcribe';

let ws, audioContext, worklet, stream;

async function startTranscription() {{
    ws = new WebSocket(WS_URL);
    ws.binaryType = 'arraybuffer';
    document.getElementById('status').textContent = 'Connecting...';

    ws.onopen = async () => {{
        document.getElementById('status').textContent = 'Connected — speak now!';
        stream       = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        audioContext = new AudioContext({{ sampleRate: SAMPLE_RATE }});
        const source = audioContext.createMediaStreamSource(stream);

        await audioContext.audioWorklet.addModule(`data:application/javascript,
            class Processor extends AudioWorkletProcessor {{
                constructor() {{ super(); this.buf = []; }}
                process(inputs) {{
                    const ch = inputs[0][0];
                    if (!ch) return true;
                    this.buf.push(...ch);
                    while (this.buf.length >= ${{FRAME_SIZE}}) {{
                        this.port.postMessage(new Float32Array(this.buf.splice(0, ${{FRAME_SIZE}})));
                    }}
                    return true;
                }}
            }}
            registerProcessor('chunker', Processor);
        `);

        worklet = new AudioWorkletNode(audioContext, 'chunker');
        source.connect(worklet);
        worklet.port.onmessage = (e) => {{
            if (ws.readyState === WebSocket.OPEN) ws.send(e.data.buffer);
        }};
    }};

    ws.onmessage = (e) => {{ document.getElementById('output').textContent += e.data; }};
    ws.onclose   = () => {{ document.getElementById('status').textContent = 'Disconnected.'; }};
    ws.onerror   = () => {{ document.getElementById('status').textContent = 'Connection error — is port 8000 open?'; }};
}}

function stopTranscription() {{
    worklet?.disconnect();
    stream?.getTracks().forEach(t => t.stop());
    audioContext?.close();
    ws?.close();
    document.getElementById('status').textContent = 'Stopped.';
}}
</script>
'''))

Server running on port 8000.


INFO:     Started server process [131636]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     149.169.80.206:4705 - "GET /transcribe HTTP/1.1" 404 Not Found


INFO:     149.169.80.206:1839 - "GET /transcribe HTTP/1.1" 404 Not Found


INFO:     149.169.80.206:4839 - "GET /transcribe HTTP/1.1" 404 Not Found


INFO:     149.169.80.206:1720 - "GET /transcribe HTTP/1.1" 404 Not Found


In [11]:
import subprocess
result = subprocess.run(['jupyter', 'server', 'list'], capture_output=True, text=True)
print(result.stdout)

INFO:     94.231.206.8:46467 - "GET / HTTP/1.1" 404 Not Found
INFO:     94.231.206.111:42503 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:40272 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:40272 - "GET /favicon.ico HTTP/1.1" 404 Not Found


In [6]:
# Debug: Find the tokenizer
print("\nCheckpoint attributes with 'token':")
print([attr for attr in dir(checkpoint_info) if 'token' in attr.lower()])

print("\nMoshi attributes with 'token':")
print([attr for attr in dir(moshi) if 'token' in attr.lower()])

print("\nMoshi.lm_model attributes with 'token' (if exists):")
if hasattr(moshi, 'lm_model'):
    print([attr for attr in dir(moshi.lm_model) if 'token' in attr.lower()])


Checkpoint attributes with 'token':
['get_text_tokenizer', 'tokenizer']

Moshi attributes with 'token':
['_get_initial_token', 'initial_token_id', 'text_initial_token_id', 'text_padding_token_id', 'ungenerated_token_id', 'zero_token_id']

Moshi.lm_model attributes with 'token' (if exists):
